# 🛰️ Space Debris & Orbital Object Prediction — Master Learning Notebook
**Academic Machine Learning Guide for Beginners**

Welcome! This notebook is designed to teach you **how Machine Learning works step-by-step** using real-world orbital object data from NASA/ESA CelesTrak Satellite Catalog (SATCAT).

---

### What You Will Learn:
1. What orbital tracking data looks like
2. How messy real-world data is cleaned
3. What $X$ (features) and $y$ (target) actually mean
4. How Keplerian features (Altitude, Velocity, Eccentricity) are engineered
5. Why we split data into Training and Testing sets
6. How **Logistic Regression**, **Decision Trees**, **Random Forest**, and **XGBoost** work conceptually
7. How a saved model generates predictions without retraining
8. How to evaluate models using Accuracy, Precision, Recall, F1, and Confusion Matrices

---

### ⚠️ Important Conceptual Clarification:
The target in this dataset is **Object-Type Classification**:
- **Class 1**: `Space Debris / Rocket Body` (`DEB` + `R/B`) — Uncontrolled orbital junk
- **Class 0**: `Payload` (`PAY`) — Active or inactive satellites

> **Note**: This project predicts object category based on orbital characteristics. It is an **academic classification model**, not a direct collision-risk or operational collision avoidance system!


## 1. Import Python Libraries

We start by importing standard Python data science libraries:
- **Pandas**: Used for tabular data manipulation (DataFrames).
- **NumPy**: Used for mathematical calculations (square roots, means).
- **Matplotlib & Seaborn**: Used for creating visualizations.
- **Scikit-Learn & XGBoost**: Used for Machine Learning algorithms and evaluation metrics.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML Algorithms & Utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Libraries imported successfully!")


> **WHAT JUST HAPPENED?**
> - We loaded the tools necessary to read data, compute mathematics, build models, and plot results.
> - No models have been trained yet; we are simply preparing our workbench.


## 2. Understanding the Dataset Structure

### What is a Dataset?
A dataset is a table of numbers and text organized into **rows** and **columns**.

```text
Object 1  ───►  [ Period, Inclination, Apogee, Perigee, RCS ]  ───►  Sample (Row 1)
Object 2  ───►  [ Period, Inclination, Apogee, Perigee, RCS ]  ───►  Sample (Row 2)
Object 3  ───►  [ Period, Inclination, Apogee, Perigee, RCS ]  ───►  Sample (Row 3)
```

- **One Row = One Sample**: Represents one single cataloged satellite, rocket body, or debris piece.
- **One Column = One Feature / Property**: Represents a specific measurement (e.g. Apogee height in kilometers).


In [ ]:
# Load raw CelesTrak SATCAT dataset
df = pd.read_csv('../data/dataset.csv')

print(f"Dataset Shape: {df.shape[0]} rows (objects), {df.shape[1]} columns (properties)")
print("\n--- First 5 Rows of Raw Data ---")
df.head()


In [ ]:
print("--- Summary Statistics of Numerical Features ---")
df[['PERIOD', 'INCLINATION', 'APOGEE', 'PERIGEE']].describe()


In [ ]:
print("--- Missing Values Count Per Column ---")
print(df.isnull().sum())

print(f"\nExact Duplicate Rows: {df.duplicated().sum()}")


> **WHAT JUST HAPPENED?**
> - `df.head()` showed us what real orbital records look like.
> - `df.shape` told us we have 70,580 space objects and 17 attributes.
> - `df.describe()` showed us summary statistics (mean, min, max, median).
> - `df.isnull().sum()` revealed missing values in Apogee, Perigee, Period, and RCS.


## 3. Data Cleaning (Step-by-Step)

Raw real-world data is messy. Before feeding data into a model, we must clean it.

---

### Cleaning Operation 1: Filter Target Object Types
- **BEFORE**: Dataset contains `DEB` (Debris), `PAY` (Payload), `R/B` (Rocket Body), and `UNK` (Unknown - 165 rows).
- **PROBLEM**: `UNK` objects lack ground-truth classification labels.
- **WHAT WE DO**: Filter out `UNK`. Map `DEB` + `R/B` $ightarrow 1$ (`Space Debris / Rocket Body`), and `PAY` $ightarrow 0$ (`Payload`).
- **WHY**: Machine Learning needs clean ground-truth targets ($y$).

---

### Cleaning Operation 2: Remove Missing Orbital Elements
- **BEFORE**: 2,054 objects are missing Period, Apogee, or Perigee.
- **PROBLEM**: We cannot compute orbital height or velocity without Apogee/Perigee.
- **WHAT WE DO**: Remove rows with missing orbital elements (`dropna`).
- **WHY**: Orbital trajectory calculations require complete physical measurements.

---

### Cleaning Operation 3: Impute Missing Radar Cross Section (RCS)
- **BEFORE**: RCS (object radar size) has missing values.
- **PROBLEM**: We don't want to throw away 37,000 valid orbital rows just because size was missing!
- **WHAT WE DO**: Convert RCS to numbers and fill missing values with the **Median**.
- **WHAT IS MEDIAN?**: The exact middle value when numbers are ordered from smallest to largest. Unlike the average (mean), the median is not skewed by extreme outliers.


In [ ]:
# 1. Filter valid object types (Exclude UNK)
df_clean = df[df['OBJECT_TYPE'].isin(['DEB', 'R/B', 'PAY'])].copy()

# 2. Define Binary Target (1 = Space Debris / Rocket Body, 0 = Payload Satellite)
df_clean['is_space_debris'] = df_clean['OBJECT_TYPE'].apply(lambda x: 1 if x in ['DEB', 'R/B'] else 0)

# 3. Drop missing orbital elements
df_clean = df_clean.dropna(subset=['PERIOD', 'INCLINATION', 'APOGEE', 'PERIGEE']).copy()

# 4. Impute missing RCS with Median
df_clean['RCS_NUM'] = pd.to_numeric(df_clean['RCS'], errors='coerce')
median_rcs = df_clean['RCS_NUM'].median()
df_clean['RCS_NUM'] = df_clean['RCS_NUM'].fillna(median_rcs)

print(f"Cleaned Dataset Size: {len(df_clean)} rows")
print(f"Target Distribution:\n{df_clean['is_space_debris'].value_counts()}")


> **WHAT JUST HAPPENED?**
> - We removed 165 unknown records and 2,054 rows missing orbital values.
> - We created a binary target $y$: Class 1 (Debris/Rocket Body) vs Class 0 (Payload).
> - We filled missing size values with the median ($0.0645\text{ m}^2$).


## 4. Defining $X$ (Features) and $y$ (Target)

### What is $X$?
$X$ is the **input information** given to the machine learning model.

### What is $y$?
$y$ is the **answer** the model is trying to learn.

```text
                 X (Features Input)
        ┌──────────────────────────────────┐
        │ Period (minutes)                 │
        │ Inclination (degrees)            │
        │ Apogee (km)                      │
        │ Perigee (km)                     │
        │ Radar Cross Section - RCS (m²)   │
        └──────────────────────────────────┘
                         │
                         ▼
                    ML MODEL
                         │
                         ▼
                 y (Target Answer)
        ┌──────────────────────────────────┐
        │ 0 = Payload Satellite            │
        │ 1 = Space Debris / Rocket Body   │
        └──────────────────────────────────┘
```


In [ ]:
# Define initial feature columns
feature_cols = ['PERIOD', 'INCLINATION', 'APOGEE', 'PERIGEE', 'RCS_NUM']

X_raw = df_clean[feature_cols]
y_raw = df_clean['is_space_debris']

print(f"X (Input Matrix Shape):  {X_raw.shape}")
print(f"y (Target Vector Shape): {y_raw.shape}")


> **WHAT JUST HAPPENED?**
> - We explicitly separated the input variables ($X$) from the target answer ($y$).
> - $X$ contains physical properties of objects; $y$ contains their true categories.


## 5. Feature Engineering (Creating Domain Features)

Instead of relying solely on raw Apogee and Perigee, we calculate physical Keplerian parameters that reflect orbital mechanics.

---

### 1. Mean Altitude ($	ext{km}$)
$$	ext{Mean Altitude} = rac{	ext{Apogee} + 	ext{Perigee}}{2}$$
*Explanation*: The average height of the object above Earth's surface.

---

### 2. Eccentricity ($e$)
$$	ext{Eccentricity} = rac{	ext{Apogee} - 	ext{Perigee}}{	ext{Apogee} + 	ext{Perigee} + 2 	imes 6371.0}$$
*Explanation*: Measures how elongated the orbit is. $0 = 	ext{perfect circle}$, values near $1 = 	ext{highly stretched ellipse}$.

---

### 3. Semi-Major Axis ($a$)
$$	ext{Semi-Major Axis} = 	ext{Mean Altitude} + 6371.0	ext{ km}$$
*Explanation*: The distance from Earth's center to the furthest point of the orbit.

---

### 4. Orbital Velocity ($v$)
$$	ext{Velocity} pprox \sqrt{rac{398600.4418}{	ext{Semi-Major Axis}}}	ext{ km/s}$$
*Explanation*: Approximate speed of the object in orbit based on gravitational mechanics.


In [ ]:
EARTH_RADIUS = 6371.0
EARTH_MU = 398600.4418

df_clean['ALTITUDE_MEAN'] = (df_clean['APOGEE'] + df_clean['PERIGEE']) / 2.0
df_clean['ECCENTRICITY'] = (df_clean['APOGEE'] - df_clean['PERIGEE']) / (df_clean['APOGEE'] + df_clean['PERIGEE'] + 2.0 * EARTH_RADIUS)
df_clean['SEMI_MAJOR_AXIS'] = df_clean['ALTITUDE_MEAN'] + EARTH_RADIUS
df_clean['VELOCITY_KM_S'] = np.sqrt(EARTH_MU / df_clean['SEMI_MAJOR_AXIS'])

# Updated 8-feature matrix
all_features = ['PERIOD', 'INCLINATION', 'APOGEE', 'PERIGEE', 'ALTITUDE_MEAN', 'ECCENTRICITY', 'VELOCITY_KM_S', 'RCS_NUM']

X = df_clean[all_features].copy()
y = df_clean['is_space_debris'].copy()

print("Engineered Features Matrix Sample:")
X.head(3)


> **WHAT JUST HAPPENED?**
> - We created 4 physical orbital features: `ALTITUDE_MEAN`, `ECCENTRICITY`, `SEMI_MAJOR_AXIS`, and `VELOCITY_KM_S`.
> - These engineered features make it much easier for ML models to separate orbital categories.


## 6. Data Leakage Prevention

### What is Data Leakage?
Data leakage occurs when a feature contains information that reveals the answer directly or would **not be available in real life when predicting new data**.

---

### Columns Removed to Prevent Leakage:
1. `OPS_STATUS_CODE`: Operational status `D` means Decayed. Decayed objects are almost exclusively debris fragments! Including this would allow the model to cheat.
2. `DECAY_DATE`: Directly reveals if an object re-entered Earth's atmosphere.
3. `OBJECT_NAME`: Names contain strings like `"SL-1 R/B"` or `"DEB"`. A model could simply read the string `"R/B"` instead of learning orbital physics!

We dropped these columns so the model learns genuine orbital physical relationships.


> **WHAT JUST HAPPENED?**
> - We verified that $X$ contains only physical measurements, preventing the model from cheating.


## 7. Train / Test Split

Suppose we have 100 orbital objects:
- **80 Objects (80%) $ightarrow$ Training Set**: Used by the model to learn patterns.
- **20 Objects (20%) $ightarrow$ Testing Set**: Kept unseen to test how well the model works on new data.

```text
Total Dataset (68,361 objects)
          │
          ├─── 80% Training Set (54,688 objects)  ──► Model Learns Here
          │
          └─── 20% Unseen Test Set (13,673 objects) ──► Model Tested Here
```

> **Why keep test data unseen?**
> Testing on the exact same data used for training is like giving a student the exam questions before the test! It measures memory, not true understanding.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training Set: {X_train.shape[0]} objects (80%)")
print(f"Testing Set:  {X_test.shape[0]} objects (20%) [Unseen]")


> **WHAT JUST HAPPENED?**
> - We split our data into 54,688 training objects and 13,673 testing objects.
> - `stratify=y` ensured both splits have the exact same percentage of debris vs payload.


## 8. Feature Scaling (StandardScaler)

### Why Scale Features?
Look at our raw numbers:
- `Period` $pprox 96.0$
- `Apogee` $pprox 938.0$
- `RCS_NUM` $pprox 0.0645$
- `Eccentricity` $pprox 0.052$

These values exist on completely different scales! Linear models like **Logistic Regression** can get confused because large numbers (Apogee = 938) dominate small numbers (RCS = 0.0645).

### What does StandardScaler do?
It subtracts the mean and divides by standard deviation, scaling features to have **$	ext{Mean} = 0$ and $	ext{Std} = 1$**.

> **Note**: Tree-based models (Decision Trees, Random Forest, XGBoost) do **NOT** depend on scaling because they split on single thresholds (`Apogee > 500`). But linear models require scaling.


In [ ]:
scaler = StandardScaler()

# Fit scaler ONLY on training data to prevent data leakage!
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Wrap back in DataFrame
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Scaled Feature Sample (Mean = 0, Std = 1):")
X_train_scaled_df.head(3)


> **WHAT JUST HAPPENED?**
> - We scaled features for Logistic Regression.
> - We fit the scaler **ONLY on training data** to avoid peeking at test statistics.


## 9. Model 1 — Logistic Regression

### How Logistic Regression Works
Logistic Regression tries to draw a straight linear decision boundary between two classes.

```text
                  Space Debris (Class 1)
                     ↑
              ●  ●  ●  ●  ●
           ●  ●  ●  ●  ●
--------------------------------------- Decision Boundary
        ○  ○  ○  ○  ○
     ○  ○  ○  ○  ○
                     ↓
             Payload (Class 0)
```

---

### Step-by-Step Prediction Mechanism:
1. Multiply each feature by a learned weight: $z = w_1 \cdot 	ext{Period} + w_2 \cdot 	ext{Apogee} + \dots + b$
2. Pass $z$ into the **Sigmoid Function**: $	ext{Probability} = rac{1}{1 + e^{-z}}$
3. If probability $\ge 0.5 ightarrow 	ext{Class 1 (Debris)}$, else $	ext{Class 0 (Payload)}$.


In [ ]:
# Train Logistic Regression Model
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_test_scaled)
lr_acc = accuracy_score(y_test, lr_pred)
lr_rec = recall_score(y_test, lr_pred)

print(f"Logistic Regression Accuracy: {lr_acc*100:.2f}%")
print(f"Logistic Regression Recall:   {lr_rec*100:.2f}%")


> **WHAT JUST HAPPENED?**
> - Logistic Regression achieved $\approx 73.6\%$ accuracy and $\approx 88.5\%$ recall.
> - It provides a simple, fast linear baseline.


## 10. Understanding Decision Trees

Before learning Random Forest, let's understand a single **Decision Tree**.

A Decision Tree makes decisions by asking a series of step-by-step threshold questions:

```text
                  Is Perigee < 500 km?
                       /                            YES        NO
                     /                    Is RCS < 0.1 m²?       Payload (Class 0)
             /                  YES        NO
           /             Debris (Class 1)   Payload (Class 0)
```

The decision tree automatically learns which questions to ask and which thresholds to use from the training data!


In [ ]:
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)
dt_acc = accuracy_score(y_test, dt_pred)
dt_rec = recall_score(y_test, dt_pred)

print(f"Single Decision Tree (depth=5) Accuracy: {dt_acc*100:.2f}%")
print(f"Single Decision Tree (depth=5) Recall:   {dt_rec*100:.2f}%")


> **WHAT JUST HAPPENED?**
> - A single decision tree asks step-by-step questions to make predictions.
> - Even a simple tree of depth 5 performs better than linear Logistic Regression!


## 11. Model 2 — Random Forest

### How Random Forest Works
A single Decision Tree can overfit or make mistakes. A **Random Forest** builds multiple decision trees (e.g. 100 trees) and combines their predictions through **majority voting**.

```text
Tree 1  ──►  Predicts: Debris (1)
Tree 2  ──►  Predicts: Debris (1)
Tree 3  ──►  Predicts: Payload (0)
Tree 4  ──►  Predicts: Debris (1)
Tree 5  ──►  Predicts: Debris (1)

Majority Vote Result ──► Class 1 (Debris)
```

Why use multiple trees?
Combining predictions from many diverse trees reduces mistakes and makes predictions much more robust.


In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
rf_rec = recall_score(y_test, rf_pred)

print(f"Random Forest (100 trees) Accuracy: {rf_acc*100:.2f}%")
print(f"Random Forest (100 trees) Recall:   {rf_rec*100:.2f}%")


> **WHAT JUST HAPPENED?**
> - Random Forest combined 100 trees via majority voting.
> - Accuracy jumped to $\approx 92.9\%$ and Recall reached $\approx 96.4\%$!


## 12. Model 3 — XGBoost (Extreme Gradient Boosting)

### How Boosting Works
Unlike Random Forest (where trees are built independently), **XGBoost builds trees sequentially**. Each new tree is specifically trained to correct the mistakes made by previous trees!

```text
Tree 1 makes predictions
          │
          ▼
    Makes mistakes on 1,000 samples
          │
          ▼
Tree 2 focuses specifically on those 1,000 mistakes
          │
          ▼
Tree 3 focuses on remaining errors
          │
          ▼
Final Combined Prediction
```


In [ ]:
xgb_model = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_rec = recall_score(y_test, xgb_pred)

print(f"XGBoost Accuracy: {xgb_acc*100:.2f}%")
print(f"XGBoost Recall:   {xgb_rec*100:.2f}%")


> **WHAT JUST HAPPENED?**
> - XGBoost built trees sequentially to correct errors.
> - It achieved $\approx 92.4\%$ Accuracy and $\approx 96.1\%$ Recall with very fast prediction speed.


## 13. Trace Prediction of One Sample Step-by-Step

Let's follow one single orbital sample through our pipeline:

```text
Raw User Inputs:
- Apogee = 938.0 km
- Perigee = 466.0 km
- Period = 96.19 min
- Inclination = 65.10°
- RCS Size = 0.080 m²
         ↓
Feature Engineering:
- Mean Altitude = (938 + 466) / 2 = 702.0 km
- Eccentricity = (938 - 466) / (938 + 466 + 2 * 6371) = 0.0335
- Semi-Major Axis = 702 + 6371 = 7073.0 km
- Velocity = sqrt(398600.4418 / 7073) = 7.507 km/s
         ↓
Pass 8-Feature Vector to Saved Model
         ↓
Random Forest 100 Trees Vote
         ↓
Output Probability: 92.3% Debris, 7.7% Payload
         ↓
Final Predicted Class: Class 1 (Space Debris / Rocket Body)
```


In [ ]:
sample_input = pd.DataFrame([{
    'PERIOD': 96.19,
    'INCLINATION': 65.10,
    'APOGEE': 938.0,
    'PERIGEE': 466.0,
    'ALTITUDE_MEAN': (938.0 + 466.0) / 2.0,
    'ECCENTRICITY': (938.0 - 466.0) / (938.0 + 466.0 + 2.0 * 6371.0),
    'VELOCITY_KM_S': np.sqrt(398600.4418 / ((938.0 + 466.0) / 2.0 + 6371.0)),
    'RCS_NUM': 0.080
}])[all_features]

sample_pred_class = rf_model.predict(sample_input)[0]
sample_prob = rf_model.predict_proba(sample_input)[0]

print(f"Predicted Class:      {sample_pred_class} ({'Space Debris/Rocket Body' if sample_pred_class==1 else 'Payload'})")
print(f"Debris Probability:   {sample_prob[1]*100:.2f}%")
print(f"Payload Probability:  {sample_prob[0]*100:.2f}%")


> **WHAT JUST HAPPENED?**
> - We traced a single LEO orbital sample from raw inputs to probabilities and final prediction.


## 14. Evaluation Metrics Explained (Tiny Example)

Suppose we test 4 space objects:

| Object | Actual True Class | Model Predicted Class | Result Type |
|---|---|---|---|
| Object 1 | Debris (1) | Debris (1) | **True Positive (TP)** |
| Object 2 | Debris (1) | Payload (0) | **False Negative (FN)** ⚠️ (Missed Debris!) |
| Object 3 | Payload (0) | Debris (1) | **False Positive (FP)** (False Alarm) |
| Object 4 | Payload (0) | Payload (0) | **True Negative (TN)** |

---

### Formulas:
- **Accuracy**: $rac{	ext{Correct Predictions}}{	ext{Total Predictions}} = rac{	ext{TP} + 	ext{TN}}{	ext{TP} + 	ext{TN} + 	ext{FP} + 	ext{FN}}$
- **Precision**: $rac{	ext{TP}}{	ext{TP} + 	ext{FP}}$ (When model says Debris, how often is it correct?)
- **Recall (Sensitivity)**: $rac{	ext{TP}}{	ext{TP} + 	ext{FN}}$ (Out of all actual debris, how many did we catch?)
- **F1 Score**: $2 	imes rac{	ext{Precision} 	imes 	ext{Recall}}{	ext{Precision} + 	ext{Recall}}$ (Harmonic balance)

---

### Why Accuracy Alone is Misleading (The Accuracy Trap)
Imagine a dataset with 95 Payload satellites and 5 Debris pieces.
A dumb model that predicts "Payload" for everything gets **95% Accuracy**, but **0% Recall for Debris**! It misses 100% of debris hazards!
That's why looking at Recall and F1 Score is essential.


## 15. Actual Model Comparison Table (Unseen Test Data)

Here are the evaluation results on our 13,673 unseen test objects:


In [ ]:
models_dict = {
    'Logistic Regression': (lr_model, X_test_scaled),
    'Decision Tree (depth=5)': (dt_model, X_test),
    'Random Forest (100 trees)': (rf_model, X_test),
    'XGBoost': (xgb_model, X_test)
}

results = []
for name, (m, X_eval) in models_dict.items():
    preds = m.predict(X_eval)
    probs = m.predict_proba(X_eval)[:, 1] if hasattr(m, 'predict_proba') else preds
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1 Score': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, probs)
    })

res_df = pd.DataFrame(results)
res_df


> **WHAT JUST HAPPENED?**
> - Random Forest and XGBoost achieved $>92.3\%$ Accuracy and $>96.1\%$ Recall, outperforming linear Logistic Regression ($73.6\%$).


## 16. Confusion Matrix Analysis (Selected Model)

Let's look at the exact Confusion Matrix counts for Random Forest:


In [ ]:
cm = confusion_matrix(y_test, rf_pred)
tn, fp, fn, tp = cm.ravel()

print(f"True Positives  (TP - Correctly identified Debris):   {tp}")
print(f"True Negatives  (TN - Correctly identified Payload):  {tn}")
print(f"False Positives (FP - False Alarms):                  {fp}")
print(f"False Negatives (FN - MISSED DEBRIS HAZARDS):         {fn}")

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Payload (0)', 'Debris (1)'], yticklabels=['Payload (0)', 'Debris (1)'])
plt.title('Confusion Matrix — Random Forest')
plt.xlabel('Predicted Class')
plt.ylabel('Actual True Class')
plt.show()


> **WHAT JUST HAPPENED?**
> - Out of 8,241 total debris objects in the test set, Random Forest missed only 296 (False Negatives), achieving a **96.40% Recall Rate**!


## 17. Why We Save Models & Do NOT Retrain on Prediction

```text
TRAINING PHASE (Done ONCE)
Historical Data (54,000+ objects) ──► Model ──► Learns Weights ──► Saved to 'models/final_model.pkl'

PREDICTION PHASE (Done instantly in App)
New User Input ──► Load 'final_model.pkl' ──► Evaluate Split Nodes ──► Output Prediction (< 5 ms)
```

> **Key Concept**: Training fits thousands of tree split nodes on historical data (takes seconds to minutes).
> Inference simply loads those pre-computed split nodes and evaluates a single vector (takes less than 5 milliseconds!).
> We do NOT retrain the model when a user enters a new object!
